In [24]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

In [34]:
soil_types = ["Sandy", "Loamy", "Clay", "Silty", "Peaty"]
crop_types = ["Wheat", "Rice", "Maize", "Sugarcane", "Cotton",
              "Chickpea", "Coconut", "Mango", "Grapes", "Banana",
              "Watermelon", "Muskmelon", "Jute", "Coffee", "Apple",
              "Mungbean", "Orange", "Papaya", "Pomegranate",
              "Blackgram", "Lentil", "Mothbeans", "Kidneybeans", "Pigeonpeas"]
fertilizers = ["Urea", "DAP", "NPK", "Potash",
               "Zinc Sulphate", "Sulphur Fertilizer",
               "Potassium Nitrate", "Organic Compost"]
growth_stages = ["Seedling", "Vegetative", "Flowering", "Harvesting"]
seasons = ["Kharif", "Rabi", "Zaid"]
zones = ["Arid", "Semi-Arid", "Humid", "Sub-Humid"]

In [35]:
def generate_row():
    soil = random.choice(soil_types)
    crop = random.choice(crop_types)
    stage = random.choice(growth_stages)
    season = random.choice(seasons)
    zone = random.choice(zones)
    N = random.randint(10, 120)
    P = random.randint(10, 100)
    K = random.randint(10, 100)
    ph = round(random.uniform(4.5, 8.5), 1)
    temp = random.uniform(15, 40)
    humidity = random.uniform(30, 90)
    rainfall = random.uniform(0, 300)
    last_fert_days = random.randint(0, 60)
    last_irrigation_days = random.randint(0, 15)
    prev_crop = random.choice(crop_types)
    # 🔥 ADVANCED DECISION LOGIC
    if last_fert_days < 10:
        fert = "Organic Compost"
    elif N < 40:
        fert = "Urea"
    elif P < 40:
        fert = "DAP"
    elif K < 40:
        fert = "Potash"
    elif ph < 5.5:
        fert = "Zinc Sulphate"
    elif rainfall < 50:
        fert = "Sulphur Fertilizer"
    elif temp > 30 and humidity < 50:
        fert = "Potassium Nitrate"
    elif stage == "Flowering":
        fert = "Potash"
    else:
        fert = "NPK"
    return [
        soil, crop, stage, season, zone,
        N, P, K, ph,
        temp, humidity, rainfall,
        last_fert_days, last_irrigation_days,
        prev_crop, fert
    ]
data = [generate_row() for _ in range(12000)]
columns = [
    "Soil", "Crop", "Stage", "Season", "Zone",
    "N", "P", "K", "pH",
    "Temp", "Humidity", "Rainfall",
    "Last_Fert_Days", "Last_Irrigation_Days",
    "Previous_Crop", "Fertilizer"
]
df = pd.DataFrame(data, columns=columns)

In [37]:
df.head()

,Soil,Crop,Stage,Season,Zone,N,P,K,pH,Temp,Humidity,Rainfall,Last_Fert_Days,Last_Irrigation_Days,Previous_Crop,Fertilizer
0,1,14,3,2,3,59,82,96,8.1,38.231999,66.994794,92.081581,46,13,5,1
1,4,20,2,0,1,12,91,66,6.7,38.376532,43.096152,78.363077,2,7,15,2
2,2,18,1,1,3,107,80,37,5.1,24.608211,53.577645,241.151089,1,0,0,2
3,1,10,2,0,2,28,52,56,8.4,33.917869,59.463606,125.965049,38,9,2,6
4,1,19,2,1,3,32,15,90,8.1,25.864406,35.044546,237.971025,10,4,13,6


In [36]:
encoders = {}
for col in ["Soil", "Crop", "Stage", "Season", "Zone", "Previous_Crop", "Fertilizer"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [38]:
X = df.drop("Fertilizer", axis=1)
y = df["Fertilizer"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestClassifier(n_estimators=400, max_depth=18)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=18, n_estimators=400)

In [39]:
joblib.dump(model, "fertilizer_model.pkl")
joblib.dump(encoders, "encoders.pkl")

['encoders.pkl']

In [40]:
def predict_fertilizer(input_dict):
    input_df = pd.DataFrame([input_dict])
    for col in ["Soil", "Crop", "Stage", "Season", "Zone", "Previous_Crop"]:
        input_df[col] = encoders[col].transform(input_df[col])
    pred = model.predict(input_df)
    return encoders["Fertilizer"].inverse_transform(pred)[0]

In [41]:
def full_recommendation(input_data):
    fert = predict_fertilizer(input_data)
    # Quantity logic
    N, P, K = input_data["N"], input_data["P"], input_data["K"]
    qty = f"{max(0, (50-N)*2)} kg/acre"
    # Timing logic
    if input_data["Stage"] == "Seedling":
        timing = "Apply before sowing"
    elif input_data["Stage"] == "Vegetative":
        timing = "Apply during early growth"
    elif input_data["Stage"] == "Flowering":
        timing = "Apply before flowering"
    else:
        timing = "Minimal application needed"
    # Method logic
    if fert in ["Urea", "DAP"]:
        method = "Broadcast + irrigation"
    elif fert == "Potassium Nitrate":
        method = "Foliar spray"
    else:
        method = "Soil application"
    # Yield estimation
    base = 20
    boost = 1.3
    yield_est = f"{base * boost:.2f} quintal/acre"
    return {
        "Fertilizer": fert,
        "Quantity": qty,
        "When": timing,
        "Method": method,
        "Expected Yield": yield_est,
        "Advice": "Balanced nutrient management will improve productivity."
    }

In [42]:
sample = {
    "Soil": "Loamy",
    "Crop": "Rice",
    "Stage": "Vegetative",
    "Season": "Kharif",
    "Zone": "Humid",
    "N": 20,
    "P": 30,
    "K": 40,
    "pH": 6.5,
    "Temp": 30,
    "Humidity": 70,
    "Rainfall": 150,
    "Last_Fert_Days": 20,
    "Last_Irrigation_Days": 3,
    "Previous_Crop": "Wheat"
}
print(full_recommendation(sample))

{'Fertilizer': 'Urea', 'Quantity': '60 kg/acre', 'When': 'Apply during early growth', 'Method': 'Broadcast + irrigation', 'Expected Yield': '26.00 quintal/acre', 'Advice': 'Balanced nutrient management will improve productivity.'}
